In [1]:
from datasets import Dataset, load_dataset

In [3]:
data_files = {"train":"//content/motivational_data.json"}
dataset = load_dataset("json", data_files=data_files)
print(data_files)
print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

{'train': '//content/motivational_data.json'}
{'prompt': 'I feel unmotivated today.', 'response': "Remember, every great journey starts with a single step. What's one small action you can take right now?"}


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"


tokenizer = AutoTokenizer.from_pretrained(model_name)


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print(f"Model loaded: {model_name}")
print(f"Vocab size: {tokenizer.vocab_size}")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded: distilgpt2
Vocab size: 50257


In [5]:
def preprocess_function(examples):
    texts = [
        f"(prompt) ### (response) ###"
        f"prompt, response in zip(examples['prompt'], examples['response'])"
    ]

    tokenized = tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    tokenized["labels"] = tokenized["input_ids"].detach().clone()

    return tokenized

In [6]:
tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)
print(tokenized_dataset["train"][0])

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

{'input_ids': [7, 16963, 457, 8, 44386, 357, 26209, 8, 44386, 16963, 457, 11, 2882, 287, 19974, 7, 1069, 12629, 17816, 16963, 457, 6, 4357, 6096, 17816, 26209, 6, 12962], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [7, 16963, 457, 8, 44386, 357, 26209, 8, 44386, 16963, 457, 11, 2882, 287, 19974, 7, 1069, 12629, 17816, 16963, 457, 6, 4357, 6096, 17816, 26209, 6, 12962]}


In [9]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./output",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    save_steps=500,
    save_total_limit=2,
    prediction_loss_only=True,
    logging_steps=10,
    learning_rate=5e-5,
    fp16=True,
    report_to="none"
)

print("training args set")

training args set


In [12]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator
)

print("Starting training")
trainer.train()
print("Training complete")

Starting training


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete


In [13]:
trainer.save_model("./final-motivational-llm")
tokenizer.save_pretrained("./final-motivational-llm")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final-motivational-llm/tokenizer_config.json',
 './final-motivational-llm/tokenizer.json')

In [14]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="./final-motivational-llm",
    tokenizer=tokenizer,
    max_new_tokens=50,
    temperature=0.7,
)

prompt = "I'm felling stuck in my routine"
output = generator(f"{prompt} ###")[0]["generated_text"]
print(output)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


I'm felling stuck in my routine ###



I'll take a lot of time to clean up my clothes before I get back in my room and leave some clothes to clean up. I'm going to be putting on a full-length jacket, a jacket, a shirt and
